# ChaiPoint Express — Sales & Business Analysis

A six-month analysis of sales, margins, customer baskets, peak hours, and outlet performance for three ChaiPoint Express cafés.



In [ ]:
import os

os.listdir()

['.ipynb_checkpoints', 'Chaiwala.ipynb', 'menu.csv', 'sales.csv']

In [ ]:
import pandas as pd

In [ ]:
import numpy as np

## 1. Data Loading



In [ ]:
sales = pd.read_csv("sales.csv")

In [ ]:
menu = pd.read_csv("menu.csv")

## 2. Initial Data Audit



In [ ]:
sales.head()

,bill_id,date,hour,outlet,item_name,quantity,price,payment_mode
0,B23683,26-04-2026,15,Karol Bagh,Masala Chai,2,30,UPI
1,B28925,09-06-2026,8,Noida Sec 18,Kulhad Chai,1,50,UPI
2,B14125,04-02-2026,9,Noida Sec 18,Masala Chai,2,30,UPI
3,B11547,14-01-2026,8,Karol Bagh,Iced Tea,1,80,UPI
4,B10511,05-01-2026,10,Karol Bagh,Samosa,1,25,UPI


In [ ]:
sales.shape


(33442, 8)

In [ ]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33442 entries, 0 to 33441
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   bill_id       33442 non-null  object
 1   date          33442 non-null  object
 2   hour          33442 non-null  int64 
 3   outlet        33442 non-null  object
 4   item_name     33442 non-null  object
 5   quantity      33442 non-null  int64 
 6   price         33442 non-null  int64 
 7   payment_mode  32940 non-null  object
dtypes: int64(3), object(5)
memory usage: 2.0+ MB


In [ ]:
sales.isnull().sum()

bill_id           0
date              0
hour              0
outlet            0
item_name         0
quantity          0
price             0
payment_mode    502
dtype: int64

In [ ]:
print("Duplicate rows :", sales.duplicated().sum())

Duplicate rows : 342


In [ ]:
print("Item spellings :", sales.item_name.nunique())

Item spellings : 64


In [ ]:
print("Negative quantity :", (sales.quantity < 0).sum() )

Negative quantity : 124


In [ ]:
print("Missing payment :", sales.payment_mode.isna().sum())

Missing payment : 502


## 3. Data Cleaning

### 3.1 Remove Duplicate Records



In [ ]:
sales = sales.drop_duplicates()

In [ ]:
sales.shape

(33100, 8)

### 3.2 Standardize Item Names



In [ ]:
sales["item_name"].unique()

array(['Masala Chai', 'Kulhad Chai', 'Iced Tea', 'Samosa',
       'Fresh Lime Soda', 'Maggi', 'Paneer Sandwich', 'Elaichi Chai',
       'Poha', 'Filter Coffee', 'Green Tea', 'Bun Maska', 'Cold Coffee',
       'Adrak Chai', 'Vada Pav', 'Aloo Patty', ' Fresh Lime Soda',
       ' Elaichi Chai', 'GREEN TEA', 'masala chai', 'ICED TEA',
       'adrak chai', 'SAMOSA', 'MAGGI', ' Adrak Chai', ' Filter Coffee',
       ' Poha', 'FRESH LIME SODA', 'MASALA CHAI', ' Masala Chai',
       'cold coffee', 'VADA PAV', 'bun maska', 'vada pav', ' Aloo Patty',
       'fresh lime soda', 'elaichi chai', 'iced tea', ' Iced Tea',
       'kulhad chai', 'ELAICHI CHAI', 'samosa', ' Cold Coffee', ' Maggi',
       ' Samosa', 'ADRAK CHAI', 'BUN MASKA', ' Vada Pav', ' Kulhad Chai',
       'FILTER COFFEE', 'COLD COFFEE', 'KULHAD CHAI', 'POHA', 'maggi',
       ' Paneer Sandwich', 'paneer sandwich', 'filter coffee',
       'ALOO PATTY', ' Bun Maska', ' Green Tea', 'green tea', 'poha',
       'PANEER SANDWICH', 'aloo pat

In [ ]:
sales["item_name"] = sales["item_name"].str.strip().str.title()

In [ ]:
sales["item_name"].nunique()

16

### 3.3 Handle Invalid Quantities



In [ ]:
sales = sales[sales.quantity > 0]

In [ ]:
sales.shape

(32979, 8)

### 3.4 Handle Missing Payment Modes



In [ ]:
sales["payment_mode"] = sales.payment_mode.fillna("Unknown")

In [ ]:
sales["payment_mode"].value_counts()

payment_mode
UPI        20147
Cash        9038
Card        3299
Unknown      495
Name: count, dtype: int64

In [ ]:
sales.columns

Index(['bill_id', 'date', 'hour', 'outlet', 'item_name', 'quantity', 'price',
       'payment_mode'],
      dtype='object')

In [ ]:
sales.columns

Index(['bill_id', 'date', 'hour', 'outlet', 'item_name', 'quantity', 'price',
       'payment_mode'],
      dtype='object')

In [ ]:
print(sales.item_name.nunique())

16


## 4. Data Preparation & Menu Merge



In [ ]:
f = sales.merge(
    menu[['item_name', 'category', 'cost']],
    on = 'item_name',
    how = 'left'
)

In [ ]:
import pandas as pd

In [ ]:
f.head()

,bill_id,date,hour,outlet,item_name,quantity,price,payment_mode,category,cost
0,B23683,26-04-2026,15,Karol Bagh,Masala Chai,2,30,UPI,Chai,15
1,B28925,09-06-2026,8,Noida Sec 18,Kulhad Chai,1,50,UPI,Chai,26
2,B14125,04-02-2026,9,Noida Sec 18,Masala Chai,2,30,UPI,Chai,15
3,B11547,14-01-2026,8,Karol Bagh,Iced Tea,1,80,UPI,Cold Beverage,30
4,B10511,05-01-2026,10,Karol Bagh,Samosa,1,25,UPI,Snack,6


In [ ]:
f.shape

(32979, 10)

## 5. Task 3 — Headline Numbers & Category Margins

### 5.1 Create Revenue and Margin Metrics



In [ ]:
f['revenue'] = f.quantity * f.price

In [ ]:
f["margin"] = f.quantity * (f.price - f.cost)

In [ ]:
f.head()

,bill_id,date,hour,outlet,item_name,quantity,price,payment_mode,category,cost,revenue,margin
0,B23683,26-04-2026,15,Karol Bagh,Masala Chai,2,30,UPI,Chai,15,60,30
1,B28925,09-06-2026,8,Noida Sec 18,Kulhad Chai,1,50,UPI,Chai,26,50,24
2,B14125,04-02-2026,9,Noida Sec 18,Masala Chai,2,30,UPI,Chai,15,60,30
3,B11547,14-01-2026,8,Karol Bagh,Iced Tea,1,80,UPI,Cold Beverage,30,80,50
4,B10511,05-01-2026,10,Karol Bagh,Samosa,1,25,UPI,Snack,6,25,19


In [ ]:
f.head()

,bill_id,date,hour,outlet,item_name,quantity,price,payment_mode,category,cost,revenue,margin
0,B23683,26-04-2026,15,Karol Bagh,Masala Chai,2,30,UPI,Chai,15,60,30
1,B28925,09-06-2026,8,Noida Sec 18,Kulhad Chai,1,50,UPI,Chai,26,50,24
2,B14125,04-02-2026,9,Noida Sec 18,Masala Chai,2,30,UPI,Chai,15,60,30
3,B11547,14-01-2026,8,Karol Bagh,Iced Tea,1,80,UPI,Cold Beverage,30,80,50
4,B10511,05-01-2026,10,Karol Bagh,Samosa,1,25,UPI,Snack,6,25,19


In [ ]:
f['gross_margin_pct'] = (f['margin']/ f['revenue'])*100

### 5.2 Headline Business Numbers



In [ ]:
f['revenue'].sum()

np.int64(1973110)

In [ ]:
f['margin'].sum()

np.int64(1180899)

In [ ]:
f['quantity'].sum()

np.int64(41140)

In [ ]:
f['margin'].sum() / f['revenue'].sum()*100

np.float64(59.84962825184606)

### 5.3 Category-Level Revenue and Profitability



In [ ]:
category_summary = f.groupby('category')['revenue'].sum()

In [ ]:
category_summary

category
Chai             659125
Coffee           154500
Cold Beverage    662630
Snack            496855
Name: revenue, dtype: int64

In [ ]:
category_profit = f.groupby('category')['margin'].sum()

In [ ]:
category_profit

category
Chai             334090
Coffee            82400
Cold Beverage    406300
Snack            358109
Name: margin, dtype: int64

In [ ]:
category_summary = f.groupby('category').agg(
    revenue = ('revenue','sum'),
    margin = ('margin','sum'),
    cost = ('cost','sum')
)

In [ ]:
category_summary['gross_margin_pct'] = (
    category_summary['margin'] / category_summary['revenue']
)*100

In [ ]:
category_summary

,revenue,margin,cost,gross_margin_pct
category,,,,
Chai,659125,334090,261507,50.686896
Coffee,154500,82400,58492,53.333333
Cold Beverage,662630,406300,204910,61.316270
Snack,496855,358109,110698,72.075153


In [ ]:
category_summary = category_summary.rename(
    columns = {'margin': 'gross_profit'}
)

In [ ]:
category_summary

,revenue,gross_profit,cost,gross_margin_pct
category,,,,
Chai,659125,334090,261507,50.686896
Coffee,154500,82400,58492,53.333333
Cold Beverage,662630,406300,204910,61.316270
Snack,496855,358109,110698,72.075153


## 6. Task 4 — Rush Hours & Staffing Analysis



# Task 4 — The Rush Hours

### 6.1 Convert Date to Datetime



In [ ]:
f["date"] = pd.to_datetime(f["date"], dayfirst = True)

In [ ]:
f["date"].head()

0   2026-04-26
1   2026-06-09
2   2026-02-04
3   2026-01-14
4   2026-01-05
Name: date, dtype: datetime64[ns]

In [ ]:
f["date"].dtype

dtype('<M8[ns]')

### 6.2 Weekday vs Weekend



In [ ]:
f["day_type"] = f["date"].dt.dayofweek.apply(
    lambda x: "Weekend" if x >= 5 else "Weekday"
)

In [ ]:
f["day_type"].value_counts()

day_type
Weekday    23268
Weekend     9711
Name: count, dtype: int64

### 6.3 Revenue by Hour



In [ ]:
hour_summary = f.groupby(["day_type", "hour"])["revenue"].sum()

In [ ]:
print(hour_summary)

day_type  hour
Weekday   8       309660
          9       160100
          10      146655
          11      112750
          12       84295
          13       71205
          14       75805
          15       79395
          16       81290
          17       76435
          18       66660
          19       49975
          20       33985
          21       20680
          22       23530
Weekend   8          790
          9         1800
          10        3635
          11        6785
          12       16105
          13       22510
          14       37825
          15       55085
          16       65495
          17       78575
          18       77215
          19       65065
          20       54740
          21       37340
          22       57725
Name: revenue, dtype: int64


In [ ]:
hour_summary = f.groupby(["day_type", "hour"])["revenue"].sum().reset_index()

In [ ]:
hour_summary

,day_type,hour,revenue
0,Weekday,8,309660
1,Weekday,9,160100
2,Weekday,10,146655
3,Weekday,11,112750
4,Weekday,12,84295
5,Weekday,13,71205
6,Weekday,14,75805
7,Weekday,15,79395
8,Weekday,16,81290
9,Weekday,17,76435


### 6.4 Identify Peak Revenue Hours



In [ ]:
hour_summary.loc[hour_summary.groupby("day_type")["revenue"].idxmax()]

,day_type,hour,revenue
0,Weekday,8,309660
24,Weekend,17,78575


### 6.5 Total Weekday and Weekend Revenue



# Calculate total weekday and weekend revenue

In [ ]:
day_summary = f.groupby("day_type")["revenue"].sum()

In [ ]:
day_summary

day_type
Weekday    1392420
Weekend     580690
Name: revenue, dtype: int64

### 6.6 Peak-Hour Revenue Share



# Calculate peak-hour revenue share

In [ ]:
309660 / 1392420 * 100

22.23897961821864

In [ ]:
78575 / 580690 * 100

13.531316192805113

### 6.7 Revenue, Volume and Profit by Outlet



# Revenue by outlet

In [ ]:
outlet_summary = f.groupby("outlet")["revenue"].sum()

In [ ]:
outlet_summary

outlet
Connaught Place    831690
Karol Bagh         668115
Noida Sec 18       473305
Name: revenue, dtype: int64

In [ ]:
outlet_quantity = f.groupby("outlet")["quantity"].sum()

In [ ]:
outlet_quantity

outlet
Connaught Place    17325
Karol Bagh         13837
Noida Sec 18        9978
Name: quantity, dtype: int64

In [ ]:
outlet_profit = f.groupby("outlet")["margin"].sum()

In [ ]:
outlet_profit

outlet
Connaught Place    497226
Karol Bagh         400627
Noida Sec 18       283046
Name: margin, dtype: int64

### 6.8 Gross Margin by Outlet



# Calculate gross margin by outlet

In [ ]:
outlet_margin = (
    f.groupby("outlet")["margin"].sum()/
    f.groupby("outlet")["revenue"].sum()
)*100

In [ ]:
outlet_margin

outlet
Connaught Place    59.785016
Karol Bagh         59.963779
Noida Sec 18       59.802030
dtype: float64

### 6.9 Time-Band Revenue Share



# Create the time bands

In [ ]:
f["time_band"] = pd.cut(
    f["hour"],
    bins = [7, 11, 14, 17, 20, 22],
    labels = ["8-11", "12-14", "15-17", "18-20", "21-22"]
)

In [ ]:
f["wknd"] = f["day_type"] == "Weekend"

In [ ]:
f[["day_type", "wknd"]].drop_duplicates()

,day_type,wknd
0,Weekend,True
1,Weekday,False


In [ ]:
band_revenue = f.groupby(
    ["time_band", "wknd"],
    observed=True
)["revenue"].sum()

In [ ]:
band_revenue

time_band  wknd 
8-11       False    729165
           True      13010
12-14      False    231305
           True      76440
15-17      False    237120
           True     199155
18-20      False    150620
           True     197020
21-22      False     44210
           True      95065
Name: revenue, dtype: int64

In [ ]:
band_pct = band_revenue.groupby(level=1).transform(
    lambda x: x / x.sum() * 100
)

In [ ]:
result = band_pct.unstack()

In [ ]:
result

wknd,False,True
time_band,,
8-11,52.366743,2.240438
12-14,16.611726,13.163650
15-17,17.029345,34.296268
18-20,10.817139,33.928602
21-22,3.175048,16.371041


## 7. Task 5 — The Combo Question

### Chai Bills Without a Snack



# Task 5 — The Combo Question (the best finding)

Question: how many chai orders leave without a snack — and what is that worth? The trick is to work at the bill level, not the row level: group by bill_id and collect the set of categories on each bill.

In [ ]:
bill_categories = f.groupby("bill_id")["category"].apply(set)

In [ ]:
chai_bill_categories = bill_categories[
    bill_categories.apply(lambda x: "Chai" in x)
]

In [ ]:
chai_bill_categories

bill_id
B10001                          {Chai}
B10002                          {Chai}
B10003    {Chai, Cold Beverage, Snack}
B10004                          {Chai}
B10006                          {Chai}
                      ...             
B31366                          {Chai}
B31367                          {Chai}
B31369           {Chai, Cold Beverage}
B31370           {Chai, Cold Beverage}
B31371                          {Chai}
Name: category, Length: 12188, dtype: object

In [ ]:
chai_no_snack = chai_bill_categories[
    ~chai_bill_categories.apply(lambda x: "Snack" in x)
]

In [ ]:
len(chai_no_snack)

8763

In [ ]:
len(chai_no_snack) / len(chai_bill_categories) * 100

71.8985887758451

### 7.1 Size the Combo Opportunity



# Size the opportunity

In [ ]:
no_snack_count = len(chai_no_snack)

In [ ]:
snack_margin = f[f["category"] == "snack"]["margin"].sum()

In [ ]:
snack_margin_per_item = (
    f[f["category"] == "Snack"]["margin"].sum()
    / f[f["category"] == "Snack"]["quantity"].sum()
)

In [ ]:
snack_margin_per_item

np.float64(30.08560867008317)

### 7.2 State the Conversion Assumption



# Assume a conversion rate
Suppose we assume that 21% of the chai-only bills would add a snack.

In [ ]:
8763 * 0.21

1840.23

In [ ]:
8763 * 0.21 * 30.08

55354.1184

In [ ]:
#If 21% of the 8,763 chai-only bills convert to a snack purchase, and each additional snack contributes about ₹30.08 gross profit, the café could generate approximately ₹55,577 additional gross profit over six months.
